#Pandas Data Cleaning 2
##Key Topics Covered:

##Duplicates

Detect duplicates with **df.duplicated()**

Remove duplicates with **df.drop_duplicates()**

**Deduplicate based on single or multiple columns** using subset

Options for **keeping first, last, or dropping all duplicates**

##Changing Data Types

Convert columns to different types using **astype()**

Safely convert numeric columns with **pd.to_numeric()**

Convert date columns to datetime using **pd.to_datetime()**

Handle missing values **(NaN) during type conversion**

##1.1 Detecting Duplicates: df.duplicated()
###Returns a Boolean Series: True for rows that are duplicates of previous rows.

By default:

All columns are considered when detecting duplicates.

**keep='first'** → marks duplicates except the first occurrence as True.

###Optional parameters:

**subset=[…] **→ check duplicates only on specific columns.

**keep='last'** → keeps the last occurrence, marks earlier duplicates True.

**keep=False** → marks all duplicates as True.

In [1]:
import pandas as pd

data = {
    'Name': ['Ali', 'Sara', 'Ali', 'Hina', 'Sara'],
    'Department': ['IT', 'HR', 'IT', 'Finance', 'HR'],
    'Salary': [50000, 60000, 50000, 55000, 60000]
}
df = pd.DataFrame(data)
df.duplicated()

,0
0,False
1,False
2,True
3,False
4,True


###Row 2 is a duplicate of row 0 (Ali, IT, 50000)

###Row 4 is a duplicate of row 1 (Sara, HR, 60000)

##1.2 Removing Duplicates: df.drop_duplicates()
Removes duplicate rows.

By default:

Keeps the first occurrence.

All columns are considered.

In [2]:
df.drop_duplicates()

,Name,Department,Salary
0,Ali,IT,50000
1,Sara,HR,60000
3,Hina,Finance,55000


###Remove duplicates based on Name column
Deduplicate customer IDs, emails, product codes, etc.

**Always check duplicates before aggregation or merging datasets.**

In [3]:
df.drop_duplicates(subset=["Name"])

,Name,Department,Salary
0,Ali,IT,50000
1,Sara,HR,60000
3,Hina,Finance,55000


##Multi-column deduplication:

In [4]:
df.drop_duplicates(subset=['Name','Department'], keep='first')

,Name,Department,Salary
0,Ali,IT,50000
1,Sara,HR,60000
3,Hina,Finance,55000


###Keep last occurrence:

In [5]:
df.drop_duplicates(subset=['Name'], keep='last')

,Name,Department,Salary
2,Ali,IT,50000
3,Hina,Finance,55000
4,Sara,HR,60000


###Drop all duplicates:

In [6]:
df.drop_duplicates(subset=['Name'], keep=False)

,Name,Department,Salary
3,Hina,Finance,55000


#2 Changing Data Types

###Goal: Ensure all columns are stored in the appropriate data type, which is critical for calculations, filtering, or machine learning.
##2.1 astype()

###Converts a column to a specified type.
####Common types: int, float, str, bool, category, Int64 (nullable integer).

###Important: If column contains **NaN**, converting to **int** will raise **IntCastingNaNError.** Use one of these strategies:

**Fill NaN before conversion:**

In [14]:
data = {
    'Name': ['Ali', 'Sara', 'Hina', 'Ahmed', 'Maaz'],
    'Age': ['25', '30', None, '22', '27'],            # Age as string + missing value
    'Salary': ['50000', '60000', '55000', None, '58000'],  # Salary as string + missing
    'JoinDate': ['2023-01-15', '2023/02/20', '15-03-2023', None, '2023-05-10'],  # Various date formats + missing
    'Department': ['IT', 'HR', 'Finance', 'IT', 'HR']
}
df1 = pd.DataFrame(data)
df1

,Name,Age,Salary,JoinDate,Department
0,Ali,25,50000,2023-01-15,IT
1,Sara,30,60000,2023/02/20,HR
2,Hina,None,55000,15-03-2023,Finance
3,Ahmed,22,None,None,IT
4,Maaz,27,58000,2023-05-10,HR


In [16]:
df1['Age'] = df1['Age'].fillna(0).astype(int)

###Can convert multiple columns at once:

In [18]:
df1[['Age','Salary']] = df1[['Age','Salary']].astype(float)

##2.2 pd.to_numeric()

###Converts a column to numeric type (int or float) safely.

###Can handle errors gracefully using errors parameter:
**errors='coerce'** → non-numeric values are converted to NaN.

**errors='ignore'** → leaves invalid values as-is.

In [21]:
pd.to_numeric(df1['Salary'],errors='coerce')

,Salary
0,50000.0
1,60000.0
2,55000.0
3,NaN
4,58000.0


In [26]:
df['Salary'] = pd.to_numeric(df1['Salary'], errors='coerce')

##2.3 pd.to_datetime()

###Converts a column to datetime type.

In [27]:
df1

,Name,Age,Salary,JoinDate,Department
0,Ali,25.0,50000.0,2023-01-15,IT
1,Sara,30.0,60000.0,2023/02/20,HR
2,Hina,0.0,55000.0,15-03-2023,Finance
3,Ahmed,22.0,NaN,None,IT
4,Maaz,27.0,58000.0,2023-05-10,HR


In [38]:
pd.to_datetime(df1['JoinDate'], errors='coerce')

,JoinDate
0,2023-01-15
1,NaT
2,NaT
3,NaT
4,2023-05-10


**You have mixed formats:**

2023-01-15 → YYYY-MM-DD

2023/02/20 → YYYY/MM/DD

15-03-2023 → DD-MM-YYYY

Option 1: Let Pandas infer automatically (no dayfirst)

Pandas automatically detects the format for each entry.

Works for most mixed formats.

In [39]:
pd.to_datetime(df1['JoinDate'], errors='coerce', infer_datetime_format = True)

/tmp/ipykernel_187/3709924260.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  pd.to_datetime(df1['JoinDate'], errors='coerce', infer_datetime_format = True)


,JoinDate
0,2023-01-15
1,NaT
2,NaT
3,NaT
4,2023-05-10


In [48]:
datam = {'JoinDate': ['2023-01-15', '2023/02/20', '15-03-2023', None]}
df2 = pd.DataFrame(datam)
df2

,JoinDate
0,2023-01-15
1,2023/02/20
2,15-03-2023
3,None


###Option 2: Standardize formats before conversion (professional approach)
###Converts / to - so Pandas can handle DD-MM-YYYY correctly.

In [49]:
df2['JoinDate'] = df2['JoinDate'].str.replace('/','-')

In [50]:
df2['JoinDate'] = pd.to_datetime(df2['JoinDate'], errors= 'coerce', dayfirst= True)
df2['JoinDate']

/tmp/ipykernel_187/1634281920.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df2['JoinDate'] = pd.to_datetime(df2['JoinDate'], errors= 'coerce', dayfirst= True)


,JoinDate
0,2023-01-15
1,2023-02-20
2,NaT
3,NaT
